# L4 10 — train faithful link

Trains only the representational adapter on neutral transmission, with frozen base weights and resumable checkpoints.

Run cells from top to bottom. Re-running resumes from Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (optional for smoke)')


In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = '4d0467233b644df641f1c5604a7b142d873d7206'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)


In [ ]:
JOB_ID = 'faithful-qwen05b-t4-001'
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
STEPS = '300'
print('Job:', JOB_ID)


In [ ]:
JOB_DIR = pathlib.Path('/content/drive/MyDrive/rival-arena-l4') / JOB_ID
JOB_DIR.mkdir(parents=True, exist_ok=True)
command = ['python', 'scripts/train_link.py', '--model', MODEL, '--output', JOB_DIR, '--job-id', JOB_ID, '--steps', str(STEPS)]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)


## Completion

The final cell exits only after the stage checkpoint is on Drive and its compact report has reached the VM receiver.